In [ ]:
# ==============================================================================
# BƯỚC 3 v12-Ablation
# Ablation study:
#   - n_last_layers: 4, 8, 16, 32
#   - Pool method: "cluster_pool" (token bị lược → gán vào cluster gần nhất
#                  trong tập token được giữ → weighted mean pool mỗi cluster)
#   - Pre-normalize vs Post-normalize cluster pool vector
#   - Scoring: traditional (sum MaxSim) và weighted (importance-weighted MaxSim)
# ==============================================================================

print(">>> BƯỚC 3 v12-Ablation")

import torch
import torch.nn.functional as F

# ==============================================================================
# CONFIG
# ==============================================================================
IMPORTANCE_VARIANT   = 'svd_sink_softplus_clusterPool_v12_ablation'
TEMPERATURE          = 0.5
MIN_IMPORTANCE_CLAMP = 1e-6
SVD_RANK_REMOVE      = 1
TOPK_RATIOS          = [round(i * 0.1, 1) for i in range(1, 10)]   # 0.1 … 0.9

# Ablation axes
N_LAST_LAYERS_LIST   = [4, 8, 16, 32]
NORMALIZE_MODES      = ['pre', 'post']

# ==============================================================================
# UTILS
# ==============================================================================
_cached_text_layers = None

def find_text_layers(model, force_rescan=False):
    global _cached_text_layers
    if _cached_text_layers is not None and not force_rescan:
        return _cached_text_layers

    candidates = []
    for name, module in model.named_modules():
        if not name.endswith('.layers') or not hasattr(module, '__len__'):
            continue
        is_vision = 'vision' in name.lower() or 'encoder' in name.lower()
        has_mlp   = hasattr(module[0], 'mlp') or hasattr(module[0], 'feed_forward')
        candidates.append({
            'name': name, 'module': module,
            'n_layers': len(module), 'is_vision': is_vision, 'has_mlp': has_mlp
        })

    text_c = [c for c in candidates if not c['is_vision'] and c['has_mlp']]
    best = max(text_c or candidates, key=lambda c: c['n_layers'], default=None)
    _cached_text_layers = best['module'] if best else None
    return _cached_text_layers


def build_content_mask(inputs, processor):
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)

    if input_ids is None:
        return attn_mask.float()

    special_ids = set()
    tok = getattr(processor, 'tokenizer', processor)

    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))

    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))

    if not special_ids:
        return attn_mask.float()

    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


# ==============================================================================
# ATTENTION COLLECTOR
# ==============================================================================
class AttentionCollector:
    def __init__(self):
        self.attentions = []
        self.hooks = []

    def _find_attn_module(self, layer):
        for attr in ['self_attn', 'attention', 'attn']:
            if hasattr(layer, attr):
                return getattr(layer, attr)
        return None

    def register_hooks(self, model, layer_indices):
        self.attentions.clear()
        self.remove_hooks()

        layers = find_text_layers(model)
        if layers is None:
            raise RuntimeError("Cannot find transformer layers")

        n_layers = len(layers)
        for idx in layer_indices:
            idx = idx if idx >= 0 else n_layers + idx
            if not (0 <= idx < n_layers):
                continue
            attn_mod = self._find_attn_module(layers[idx])
            if attn_mod is None:
                continue

            def hook(module, inp, out):
                if isinstance(out, tuple) and len(out) >= 2:
                    attn = out[1]
                    if attn is not None:
                        self.attentions.append(attn.detach())

            self.hooks.append(attn_mod.register_forward_hook(hook))

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks.clear()

    def clear(self):
        self.attentions.clear()


# ==============================================================================
# SVD CORE — batched over all heads at once
# ==============================================================================
def remove_sink_components_batch(attn_heads, k):
    try:
        U, S, Vh = torch.linalg.svd(attn_heads, full_matrices=False)
        k = min(k, S.shape[-1])
        sink = (U[..., :k] * S[:, :k].unsqueeze(1)) @ Vh[:, :k, :]
        return (attn_heads - sink).clamp(min=0.0)
    except Exception:
        return attn_heads


# ==============================================================================
# IMPORTANCE COMPUTATION — SOFTPLUS  (was: SIGMOID)
# ==============================================================================
def compute_svd_importance_softplus(attentions, content_mask, layer_weights, k):
    device = content_mask.device
    B, S = content_mask.shape
    importance = torch.zeros(B, S, device=device)

    for i, attn in enumerate(attentions):
        attn = attn.float().to(device)
        B_, H, Sq, Sk = attn.shape

        attn_flat = attn.view(B_ * H, Sq, Sk)
        cleaned   = remove_sink_components_batch(attn_flat, k)
        cleaned   = cleaned.view(B_, H, Sq, Sk)

        layer_imp = cleaned.sum(dim=2).mean(dim=1)
        layer_imp = layer_imp * content_mask
        layer_imp = layer_imp / layer_imp.max(dim=-1, keepdim=True).values.clamp(min=1e-8)
        importance += layer_weights[i] * layer_imp

    importance = importance * content_mask
    # --- CHANGED: sigmoid → softplus ---
    importance = F.softplus(importance / TEMPERATURE)
    importance = importance * content_mask
    return importance


# ==============================================================================
# DOC MATRIX BUILDER
# ==============================================================================
def build_doc_matrix(docs_emb, device):
    n_docs  = len(docs_emb)
    max_len = max(d.shape[0] for d in docs_emb)
    D       = docs_emb[0].shape[1]

    doc_matrix = torch.zeros(n_docs, max_len, D,    device=device)
    doc_mask   = torch.zeros(n_docs, max_len,        device=device, dtype=torch.bool)

    for i, d in enumerate(docs_emb):
        L = d.shape[0]
        doc_matrix[i, :L] = F.normalize(d.float().to(device), dim=-1)
        doc_mask[i, :L]   = True

    return doc_matrix, doc_mask


# ==============================================================================
# FAST MAXSIM
# ==============================================================================
@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values   # (S_q, n_docs)


# ==============================================================================
# CLUSTER-POOL BUILDER
# ==============================================================================
def build_cluster_pool_scores(
    q_norm_kept, q_norm_disc,
    imp_kept, imp_disc,
    doc_matrix, doc_mask,
    normalize_mode='pre',
):
    K, D   = q_norm_kept.shape
    P      = q_norm_disc.shape[0]
    n_docs = doc_matrix.shape[0]

    if P == 0:
        return torch.zeros(n_docs, device=doc_matrix.device), 0.0

    sim_assign  = torch.mm(q_norm_disc, q_norm_kept.t())
    cluster_ids = sim_assign.argmax(dim=-1)

    cluster_scores = torch.zeros(n_docs, device=doc_matrix.device)
    total_disc_imp = imp_disc.sum().item()

    for c in cluster_ids.unique():
        members  = (cluster_ids == c).nonzero(as_tuple=True)[0]
        w        = imp_disc[members]
        w_sum    = w.sum().clamp(min=1e-8)
        w_norm   = w / w_sum
        pool_vec = (q_norm_disc[members] * w_norm.unsqueeze(-1)).sum(dim=0)

        if normalize_mode == 'pre':
            pool_vec = F.normalize(pool_vec.unsqueeze(0), dim=-1)
            sim = torch.einsum('qd,nld->qnl', pool_vec, doc_matrix)
            sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
            ms = sim.max(dim=-1).values.squeeze(0)
            cluster_scores += ms * w_sum.item()
        else:
            pool_vec_u = pool_vec.unsqueeze(0)
            sim = torch.einsum('qd,nld->qnl', pool_vec_u, doc_matrix)
            sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
            ms = sim.max(dim=-1).values.squeeze(0)
            cluster_scores += ms * w_sum.item()

    if total_disc_imp > 1e-8:
        cluster_scores = cluster_scores / total_disc_imp

    return cluster_scores, total_disc_imp


# ==============================================================================
# MAIN EXTRACTION FUNCTION v12
# ==============================================================================
def extract_embeddings_and_importance_v12(model, inputs, processor, n_last_layers=4):
    collector = AttentionCollector()
    collector.register_hooks(model, list(range(-n_last_layers, 0)))

    try:
        with torch.no_grad():
            outputs = model(**inputs, output_attentions=True, return_dict=True)

            if hasattr(outputs, "last_hidden_state"):
                proj = outputs.last_hidden_state
            elif isinstance(outputs, torch.Tensor):
                proj = outputs
            else:
                raise RuntimeError("Unknown model output format")

            proj = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)
            proj = proj * inputs["attention_mask"].unsqueeze(-1).float()

        content_mask = build_content_mask(inputs, processor).to(proj.device)

        attn_list = (
            list(outputs.attentions[-n_last_layers:])
            if hasattr(outputs, "attentions") and outputs.attentions is not None
            else collector.attentions
        )
        n_actual = len(attn_list)

        layer_weights = torch.exp(
            torch.linspace(0, 1, max(n_actual, 1), device=proj.device)
        )
        layer_weights /= layer_weights.sum()

        if n_actual > 0:
            importance = compute_svd_importance_softplus(
                attn_list, content_mask, layer_weights, SVD_RANK_REMOVE
            )
        else:
            importance = content_mask.clone()

        importance = importance * content_mask

        collector.remove_hooks()
        collector.clear()
        return proj, importance, (n_actual > 0)

    except Exception as e:
        print(f"❌ Extraction error: {e}")
        collector.remove_hooks()
        collector.clear()

        with torch.no_grad():
            outputs = model(**inputs, return_dict=True)
            proj = outputs.last_hidden_state
            proj = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)
            proj = proj * inputs["attention_mask"].unsqueeze(-1).float()

        content_mask = build_content_mask(inputs, processor).to(proj.device)
        importance   = content_mask.clone()
        return proj, importance, False


print(">>> BƯỚC 3 v12-Ablation COMPLETE")


# ==============================================================================
# BƯỚC 3.5: UTILITY FUNCTIONS
# ==============================================================================
print(">>> BƯỚC 3.5: Loading utility functions...")

import numpy as np


def compute_ndcg(ranked_indices, gt_set, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, idx in enumerate(ranked_indices[:k]) if idx in gt_set)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt_set), k)))
    return dcg / idcg if idcg > 0 else 0.0


def first_hit(top_k, gt_set):
    for r, idx in enumerate(top_k):
        if idx in gt_set:
            return r + 1
    return -1


def pretty_token(t):
    t = str(t).replace("\n", "\\n").replace(" ", "_")
    return t if len(t) <= 24 else (t[:22] + "..")


print("✅ Utility functions loaded")
print(">>> BƯỚC 3.5 COMPLETE")


# ==============================================================================
# BƯỚC 4: COMPREHENSIVE BATCH EVALUATION — ABLATION
# ==============================================================================
print(">>> BƯỚC 4: Process All Batches — Ablation Study")
print(f"Processing {len(BATCH_RANGES)} document batches: {BATCH_RANGES}")
print(f"n_last_layers variants : {N_LAST_LAYERS_LIST}")
print(f"normalize_mode variants: {NORMALIZE_MODES}")
print(f"TOPK_RATIOS            : {TOPK_RATIOS}")

import json
import os
import pickle
import gc
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

WORKING_DIR       = "/kaggle/working"
ANNOTATIONS_PATH  = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
COLSMOL_DIR       = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
QUERY_BATCH_SIZE  = 50


# ==============================================================================
# Build method key list
# ==============================================================================
def make_ablation_keys():
    keys = ['traditional', 'trad_weighted']
    for n in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for r in TOPK_RATIOS:
                tag = f"L{n}_norm{mode}_r{int(r*100)}"
                keys.append(f"{tag}_trad")
                keys.append(f"{tag}_weighted")
    return keys

ABLATION_KEYS = make_ablation_keys()


# ==============================================================================
# METRIC HELPERS
# NOTE: _init_metric returns a plain dict of Python scalars only.
#       Never store nested dicts inside it.
# ==============================================================================
def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0, 'n1': 0.0, 'n5': 0.0, 'n10': 0.0, 'count': 0}


def _add_metric(dst, src):
    """Add scalar values from hit_metrics() result src into accumulator dst."""
    dst['r1']    += int(src['r1'])
    dst['r5']    += int(src['r5'])
    dst['r10']   += int(src['r10'])
    dst['n1']    += float(src['n1'])
    dst['n5']    += float(src['n5'])
    dst['n10']   += float(src['n10'])
    dst['count'] += 1


def _ensure(store, key):
    """Return store[key], creating it as _init_metric() if absent."""
    if key not in store:
        store[key] = _init_metric()
    return store[key]


def hit_metrics(top10, gt):
    h = first_hit(top10, gt)
    return {
        'r1':  int(h != -1 and h <= 1),
        'r5':  int(h != -1 and h <= 5),
        'r10': int(h != -1 and h <= 10),
        'n1':  float(compute_ndcg(top10, gt, 1)),
        'n5':  float(compute_ndcg(top10, gt, 5)),
        'n10': float(compute_ndcg(top10, gt, 10)),
    }


def record(key, m, domain):
    """Record hit_metrics result m into global and per-domain stores."""
    _add_metric(_ensure(all_metrics,               key), m)
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


# ==============================================================================
# MASTER STORAGE
# ==============================================================================
all_query_results  = []
all_token_results  = []
all_metrics        = {}   # key → _init_metric() dict
all_domain_metrics = {}   # domain → key → _init_metric() dict
all_batch_stats    = []

print("\n" + "=" * 80)
print("BATCH LOOP PROCESSING")
print("=" * 80)

for batch_num, (START_IDX, END_IDX) in enumerate(BATCH_RANGES, 1):
    print(f"\n[{batch_num}/{len(BATCH_RANGES)}] Processing Batch [{START_IDX}:{END_IDX}]")

    # =========================================================================
    # LOAD INDEX
    # =========================================================================
    index_path = os.path.join(COLSMOL_DIR, f"{START_IDX}-{END_IDX}.pkl")
    if not os.path.exists(index_path):
        print(f"  ⚠️  Index not found: {index_path}")
        continue

    try:
        with open(index_path, 'rb') as f:
            saved = pickle.load(f)
        if isinstance(saved, dict):
            fused_embeddings = saved.get('embeddings', [])
            fused_importance = saved.get('importance', None)
            has_importance   = fused_importance is not None
        else:
            fused_embeddings = saved
            fused_importance = None
            has_importance   = False
        print(f"  ✅ Loaded index: {len(fused_embeddings)} layouts, importance={has_importance}")
    except Exception as e:
        print(f"  ❌ Error loading index: {e}")
        continue

    # =========================================================================
    # LOAD BATCH DATA
    # =========================================================================
    print("  Loading batch data...")
    try:
        batch_doc_names    = intersection_docs[START_IDX:END_IDX]
        batch_target_files = [jsonl_map[d] for d in batch_doc_names]

        batch_df_orig = pd.read_parquet(PARQUET_PATH)
        batch_df_orig['join_doc_name'] = batch_df_orig['doc_name'].str.replace('.pdf', '', regex=False)
        batch_df_orig = batch_df_orig[batch_df_orig['join_doc_name'].isin(batch_doc_names)]

        dfs = []
        for f in tqdm(batch_target_files, desc="    Reading JSONLs", leave=False):
            try:
                temp = pd.read_json(f, lines=True)
                temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
                if 'layout' in temp.columns:
                    temp = temp.rename(columns={'layout': 'layout_id'})
                cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
                if 'text_level' in temp.columns:
                    cols.append('text_level')
                temp = temp[[c for c in cols if c in temp.columns]]
                dfs.append(temp)
            except Exception:
                pass

        if dfs:
            batch_df_enh = pd.concat(dfs, ignore_index=True)
            batch_df_enh = batch_df_enh.rename(columns={
                'vlm_text': 'vlm_text_enhanced',
                'text_level': 'text_level_enhanced'
            })
        else:
            batch_df_enh = pd.DataFrame()

        batch_df_final = pd.merge(
            batch_df_orig, batch_df_enh,
            on=['join_doc_name', 'layout_id'], how='left'
        )
        batch_df_final = batch_df_final.sort_values(by=['join_doc_name', 'page_id', 'layout_id'])

        is_header_type = batch_df_final['type'].isin(['title', 'section_header', 'header'])
        has_text_level = batch_df_final.get('text_level_enhanced', pd.Series(dtype=object)).notna()
        is_header = is_header_type | has_text_level
        batch_df_final['temp_header'] = batch_df_final['text'].where(is_header)
        batch_df_final['current_section'] = (
            batch_df_final.groupby('join_doc_name')['temp_header']
            .ffill().fillna("General Content")
        )

        enh_image_map = {}
        if os.path.exists(ENHANCED_IMG_DIR):
            for fn in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
                enh_image_map[os.path.basename(fn)] = fn

        df = batch_df_final.copy()
        if 'img_enhanced_path' in df.columns:
            img_fnames = df['img_enhanced_path'].dropna().map(
                lambda p: enh_image_map.get(os.path.basename(str(p)))
            )
            df['img_data'] = img_fnames
            df['img_type'] = df['img_data'].notna().map(lambda x: 'path' if x else None)
        else:
            df['img_data'] = None
            df['img_type'] = None

        has_binary = 'image_binary' in df.columns
        if has_binary:
            need_fallback = df['img_type'].isna() & df['image_binary'].notna()
            df.loc[need_fallback, 'img_type'] = 'binary'
            df.loc[need_fallback, 'img_data'] = df.loc[need_fallback, 'image_binary']

        def _pick_text(row):
            for col in ['vlm_text_enhanced', 'text', 'ocr_text', 'vlm_text']:
                v = row.get(col)
                if pd.notna(v) and len(str(v)) > 5:
                    return str(v)
            return "Document layout."

        raw_content = df.apply(_pick_text, axis=1)
        df['final_text'] = "Section: " + df['current_section'].fillna('') + " \n Content: " + raw_content

        batch_sample_layouts_df = df.dropna(subset=['img_type']).reset_index(drop=True)
        print(f"  ✅ Batch data: {len(batch_sample_layouts_df)} layouts")

    except Exception as e:
        print(f"  ❌ Error loading batch data: {e}")
        import traceback; traceback.print_exc()
        continue

    # =========================================================================
    # BUILD QA PAIRS
    # =========================================================================
    print("  Building QA pairs...")

    def calculate_iou(box1, box2):
        b1 = list(box1) if not isinstance(box1, list) else box1
        b2 = list(box2) if not isinstance(box2, list) else box2
        x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
        x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
        inter  = max(0, x2 - x1) * max(0, y2 - y1)
        union  = ((b1[2]-b1[0])*(b1[3]-b1[1])) + ((b2[2]-b2[0])*(b2[3]-b2[1])) - inter
        return inter / union if union > 0 else 0.0

    batch_qa_pairs   = []
    batch_doc_lookup = {k: v for k, v in batch_sample_layouts_df.groupby('join_doc_name')}
    batch_avail_docs = set(batch_doc_lookup.keys())

    with open(ANNOTATIONS_PATH, 'r') as f:
        for line in f:
            try:
                doc_data = json.loads(line)
            except Exception:
                continue
            target_doc = doc_data['doc_name'].replace('.pdf', '')
            if target_doc not in batch_avail_docs:
                continue

            doc_layouts    = batch_doc_lookup[target_doc]
            domain         = doc_data.get('domain', 'General')
            col_page       = 'page_idx' if 'page_idx' in doc_layouts.columns else 'page_id'
            current_doc_df = doc_layouts.copy()
            current_doc_df['safe_page'] = (
                pd.to_numeric(current_doc_df[col_page], errors='coerce')
                .fillna(-999).astype(int)
            )

            for q_item in doc_data.get('questions', []):
                gt_indices = []
                if 'layout_mapping' in q_item:
                    for target in q_item['layout_mapping']:
                        t_page, t_bbox = target['page'], target['bbox']
                        try:
                            t_page_int = int(t_page)
                            cands = pd.concat([
                                current_doc_df[current_doc_df['safe_page'] == t_page_int],
                                current_doc_df[current_doc_df['safe_page'] == t_page_int - 1],
                            ])
                            for idx, row in cands.iterrows():
                                if calculate_iou(row['bbox'], t_bbox) > 0.5:
                                    gt_indices.append(int(idx))
                        except Exception:
                            continue
                if gt_indices:
                    batch_qa_pairs.append({
                        'question':   q_item['Q'],
                        'gt_indices': list(set(gt_indices)),
                        'doc_name':   target_doc,
                        'domain':     domain,
                    })

    print(f"  ✅ Found {len(batch_qa_pairs)} QA pairs")

    if len(batch_qa_pairs) == 0:
        print("  ⚠️  No QA pairs, skipping")
        all_batch_stats.append({
            'batch': f"{START_IDX}-{END_IDX}",
            'n_layouts': len(fused_embeddings),
            'n_qa': 0, 'status': 'no_qa'
        })
        continue

    # =========================================================================
    # EVALUATE — ABLATION
    # =========================================================================
    print(f"  Evaluating {len(batch_qa_pairs)} queries across "
          f"{len(N_LAST_LAYERS_LIST)} n_last_layers × "
          f"{len(NORMALIZE_MODES)} norm_modes × "
          f"{len(TOPK_RATIOS)} ratios × 2 scoring modes ...")

    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        n_docs = len(fused_embeddings)
        total  = len(batch_qa_pairs)

        print("  Building doc matrix (once)...")
        doc_matrix, doc_mask = build_doc_matrix(fused_embeddings, device)
        print(f"  ✅ Doc matrix: {doc_matrix.shape}")

        batch_query_rows = []
        batch_token_rows = []

        # Local batch-level counters (reset each batch, used only for printing)
        batch_metrics = {k: _init_metric() for k in ABLATION_KEYS}

        for query_batch_start in range(0, total, QUERY_BATCH_SIZE):
            query_batch_end   = min(query_batch_start + QUERY_BATCH_SIZE, total)
            query_batch_range = batch_qa_pairs[query_batch_start:query_batch_end]

            pbar = tqdm(
                enumerate(query_batch_range),
                total=len(query_batch_range),
                desc=f"    Q[{query_batch_start}:{query_batch_end}]",
                leave=False
            )

            for local_q_idx, item in pbar:
                global_q_idx = query_batch_start + local_q_idx
                question     = item['question']
                gt_set       = set(item['gt_indices'])
                domain       = item['domain']

                # Ensure domain sub-dict exists BEFORE any record() call
                if domain not in all_domain_metrics:
                    all_domain_metrics[domain] = {}

                with torch.no_grad():
                    q_inputs = processor.process_queries([question]).to(device)

                attn_mask       = q_inputs['attention_mask'][0].float()
                content_mask_1d = build_content_mask(q_inputs, processor)[0].float()

                trad_idx   = torch.where(attn_mask > 0)[0]
                method_idx = torch.where(content_mask_1d > 0)[0]
                if trad_idx.numel() == 0:
                    continue
                if method_idx.numel() == 0:
                    method_idx = trad_idx

                # ----------------------------------------------------------
                # Run model once per n_last_layers value — cache results
                # ----------------------------------------------------------
                embed_cache = {}
                for n_layers in N_LAST_LAYERS_LIST:
                    proj, imp, _ = extract_embeddings_and_importance_v12(
                        model, q_inputs, processor, n_last_layers=n_layers
                    )
                    embed_cache[n_layers] = (proj[0].float(), imp[0].float())

                # ----------------------------------------------------------
                # BASELINE: Traditional MaxSim
                # ----------------------------------------------------------
                q_embed_base = embed_cache[N_LAST_LAYERS_LIST[0]][0]
                q_trad_norm  = F.normalize(q_embed_base[trad_idx].float(), dim=-1)
                M_trad       = fast_maxsim(q_trad_norm, doc_matrix, doc_mask)

                trad_scores  = M_trad.sum(dim=0)
                trad_top10   = torch.topk(trad_scores, k=min(10, n_docs)).indices.cpu().tolist()
                trad_m       = hit_metrics(trad_top10, gt_set)
                _add_metric(batch_metrics['traditional'], trad_m)
                record('traditional', trad_m, domain)

                # BASELINE: Traditional importance-weighted
                imp_base      = embed_cache[N_LAST_LAYERS_LIST[0]][1]
                imp_trad      = imp_base[trad_idx]
                imp_trad_n    = imp_trad / imp_trad.sum().clamp(min=1e-8)
                trad_w_scores = (M_trad * imp_trad_n.unsqueeze(-1)).sum(dim=0)
                trad_w_top10  = torch.topk(trad_w_scores, k=min(10, n_docs)).indices.cpu().tolist()
                trad_w_m      = hit_metrics(trad_w_top10, gt_set)
                _add_metric(batch_metrics['trad_weighted'], trad_w_m)
                record('trad_weighted', trad_w_m, domain)

                query_row = {
                    'batch':         f"{START_IDX}-{END_IDX}",
                    'query_id':      global_q_idx,
                    'doc_name':      item['doc_name'],
                    'domain':        item['domain'],
                    'question':      question,
                    'trad_r@1':      trad_m['r1'],
                    'trad_r@5':      trad_m['r5'],
                    'trad_r@10':     trad_m['r10'],
                    'trad_ndcg@1':   round(trad_m['n1'],  4),
                    'trad_ndcg@5':   round(trad_m['n5'],  4),
                    'trad_ndcg@10':  round(trad_m['n10'], 4),
                    'tradW_r@1':     trad_w_m['r1'],
                    'tradW_r@5':     trad_w_m['r5'],
                    'tradW_r@10':    trad_w_m['r10'],
                    'tradW_ndcg@1':  round(trad_w_m['n1'],  4),
                    'tradW_ndcg@5':  round(trad_w_m['n5'],  4),
                    'tradW_ndcg@10': round(trad_w_m['n10'], 4),
                }

                # ----------------------------------------------------------
                # ABLATION LOOP: n_last_layers × normalize_mode × ratio × scoring
                # ----------------------------------------------------------
                for n_layers in N_LAST_LAYERS_LIST:
                    q_embed, q_importance = embed_cache[n_layers]

                    q_method_norm  = F.normalize(q_embed[method_idx].float(), dim=-1)
                    M_full         = fast_maxsim(q_method_norm, doc_matrix, doc_mask)

                    imp_valid      = q_importance[method_idx].float()
                    n_method       = method_idx.numel()
                    sorted_imp_idx = torch.argsort(imp_valid, descending=True)

                    for mode in NORMALIZE_MODES:
                        for r in TOPK_RATIOS:
                            tag    = f"L{n_layers}_norm{mode}_r{int(r*100)}"
                            n_keep = max(1, int(n_method * r))

                            kept_idx = sorted_imp_idx[:n_keep]
                            disc_idx = sorted_imp_idx[n_keep:]

                            q_kept   = q_method_norm[kept_idx]
                            q_disc   = q_method_norm[disc_idx]
                            imp_kept = imp_valid[kept_idx]
                            imp_disc = imp_valid[disc_idx]
                            M_kept   = M_full[kept_idx]

                            cluster_scores, total_disc_imp = build_cluster_pool_scores(
                                q_kept, q_disc, imp_kept, imp_disc,
                                doc_matrix, doc_mask, normalize_mode=mode,
                            )
                            pool_frac = total_disc_imp / imp_valid.sum().clamp(min=1e-8).item()

                            # SCORING MODE 1: Traditional sum
                            scores_trad = M_kept.sum(dim=0) + cluster_scores * pool_frac
                            top10_trad  = torch.topk(scores_trad, k=min(10, n_docs)).indices.cpu().tolist()
                            m_trad      = hit_metrics(top10_trad, gt_set)
                            key_trad    = f"{tag}_trad"
                            _add_metric(batch_metrics[key_trad], m_trad)
                            record(key_trad, m_trad, domain)
                            query_row.update({
                                f'{key_trad}_r@1':     m_trad['r1'],
                                f'{key_trad}_r@5':     m_trad['r5'],
                                f'{key_trad}_r@10':    m_trad['r10'],
                                f'{key_trad}_ndcg@1':  round(m_trad['n1'],  4),
                                f'{key_trad}_ndcg@5':  round(m_trad['n5'],  4),
                                f'{key_trad}_ndcg@10': round(m_trad['n10'], 4),
                            })

                            # SCORING MODE 2: Importance-weighted
                            imp_kept_n  = imp_kept / imp_kept.sum().clamp(min=1e-8)
                            scores_wt   = (M_kept * imp_kept_n.unsqueeze(-1)).sum(dim=0)
                            scores_wt   = scores_wt + cluster_scores * pool_frac
                            top10_wt    = torch.topk(scores_wt, k=min(10, n_docs)).indices.cpu().tolist()
                            m_wt        = hit_metrics(top10_wt, gt_set)
                            key_wt      = f"{tag}_weighted"
                            _add_metric(batch_metrics[key_wt], m_wt)
                            record(key_wt, m_wt, domain)
                            query_row.update({
                                f'{key_wt}_r@1':     m_wt['r1'],
                                f'{key_wt}_r@5':     m_wt['r5'],
                                f'{key_wt}_r@10':    m_wt['r10'],
                                f'{key_wt}_ndcg@1':  round(m_wt['n1'],  4),
                                f'{key_wt}_ndcg@5':  round(m_wt['n5'],  4),
                                f'{key_wt}_ndcg@10': round(m_wt['n10'], 4),
                            })

                batch_query_rows.append(query_row)

                # Token-level rows (reference: first n_last_layers value)
                q_imp_ref  = embed_cache[N_LAST_LAYERS_LIST[0]][1]
                q_emb_ref  = embed_cache[N_LAST_LAYERS_LIST[0]][0]
                q_ref_norm = F.normalize(q_emb_ref[method_idx].float(), dim=-1)
                M_ref      = fast_maxsim(q_ref_norm, doc_matrix, doc_mask)

                method_idx_np           = method_idx.cpu().numpy()
                imp_np                  = q_imp_ref[method_idx].cpu().numpy()
                input_ids               = q_inputs['input_ids'][0].cpu().tolist()
                tok_str_all             = processor.tokenizer.convert_ids_to_tokens(input_ids)
                global_token_best_score = M_ref.max(dim=1).values.cpu().numpy()

                for local_t, global_t in enumerate(method_idx_np):
                    tok_raw = (tok_str_all[int(global_t)]
                               if int(global_t) < len(tok_str_all) else "<UNK>")
                    batch_token_rows.append({
                        'batch':      f"{START_IDX}-{END_IDX}",
                        'query_id':   global_q_idx,
                        'token_idx':  int(global_t),
                        'token':      pretty_token(tok_raw),
                        'importance': round(float(imp_np[local_t]), 8),
                        'maxsim_gt':  round(float(global_token_best_score[local_t]), 8),
                    })

            gc.collect()
            torch.cuda.empty_cache()
            print(f"      ✓ Q[{query_batch_start}:{query_batch_end}] done")

        all_query_results.extend(batch_query_rows)
        all_token_results.extend(batch_token_rows)

        t_cnt    = batch_metrics['traditional']['count']
        t_r10    = batch_metrics['traditional']['r10'] / t_cnt * 100 if t_cnt else 0
        t_ndcg10 = batch_metrics['traditional']['n10'] / t_cnt       if t_cnt else 0
        print(f"  ✅ Batch done: {len(batch_query_rows)} Q, {len(batch_token_rows)} T")
        print(f"     Baseline R@10={t_r10:.1f}%  nDCG@10={t_ndcg10:.4f}")

        all_batch_stats.append({
            'batch':          f"{START_IDX}-{END_IDX}",
            'n_layouts':      len(fused_embeddings),
            'n_queries':      total,
            'trad_recall@10': round(t_r10,    2),
            'trad_ndcg@10':   round(t_ndcg10, 4),
            'status':         'ok'
        })

    except Exception as e:
        print(f"  ❌ Evaluation error: {e}")
        import traceback; traceback.print_exc()
        all_batch_stats.append({
            'batch': f"{START_IDX}-{END_IDX}",
            'status': 'error', 'error': str(e)
        })

    # =========================================================================
    # SAVE & CLEANUP
    # =========================================================================
    try:
        if batch_query_rows:
            pd.DataFrame(batch_query_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_queries.csv"),
                index=False
            )
        if batch_token_rows:
            pd.DataFrame(batch_token_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_tokens.csv"),
                index=False
            )
        print("  ✅ Batch files saved")
    except Exception as e:
        print(f"  ❌ Save error: {e}")

    del doc_matrix, doc_mask
    del fused_embeddings, batch_sample_layouts_df, batch_qa_pairs
    del batch_df_orig, batch_df_enh, batch_df_final
    gc.collect()
    torch.cuda.empty_cache()
    print("  ✓ Batch memory cleaned\n")


# ==============================================================================
# FINAL: CONSOLIDATE & SUMMARY
# ==============================================================================
print("=" * 80)
print("CONSOLIDATING RESULTS")
print("=" * 80)

try:
    if all_query_results:
        df_all_q = pd.DataFrame(all_query_results)
        master_q = os.path.join(WORKING_DIR, "MASTER_all_batches_queries.csv")
        df_all_q.to_csv(master_q, index=False)
        print(f"✅ {len(df_all_q)} queries → {master_q}")

    if all_token_results:
        df_all_t = pd.DataFrame(all_token_results)
        master_t = os.path.join(WORKING_DIR, "MASTER_all_batches_tokens.csv")
        df_all_t.to_csv(master_t, index=False)
        print(f"✅ {len(df_all_t)} tokens → {master_t}")

    total_q = sum(
        row.get('n_queries', 0)
        for row in all_batch_stats if row.get('status') == 'ok'
    )

    # =========================================================================
    # OVERALL SUMMARY TABLE
    # =========================================================================
    print(f"\n{'Method':<42} {'R@1':>7} {'R@5':>7} {'R@10':>7} "
          f"{'nDCG@1':>9} {'nDCG@5':>9} {'nDCG@10':>9}")
    print("-" * 100)

    for method in ['traditional', 'trad_weighted']:
        if method not in all_metrics:
            continue
        m   = all_metrics[method]
        cnt = m['count'] if m['count'] > 0 else 1
        print(f"{method:<42} "
              f"{m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  {m['r10']/cnt*100:6.2f}%  "
              f"{m['n1']/cnt:8.4f}  {m['n5']/cnt:8.4f}  {m['n10']/cnt:8.4f}")

    print("-" * 100)

    for n_layers in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for scoring in ['trad', 'weighted']:
                print()
                for r in TOPK_RATIOS:
                    tag = f"L{n_layers}_norm{mode}_r{int(r*100)}_{scoring}"
                    if tag not in all_metrics:
                        continue
                    m   = all_metrics[tag]
                    cnt = m['count'] if m['count'] > 0 else 1
                    print(f"{tag:<42} "
                          f"{m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  {m['r10']/cnt*100:6.2f}%  "
                          f"{m['n1']/cnt:8.4f}  {m['n5']/cnt:8.4f}  {m['n10']/cnt:8.4f}")

    # =========================================================================
    # PER-DOMAIN BREAKDOWN
    # =========================================================================
    print("\n" + "=" * 130)
    print("PER-DOMAIN BREAKDOWN — BASELINE vs TRAD-WEIGHTED")
    print("=" * 130)

    all_domains = sorted(all_domain_metrics.keys())

    print(f"\n{'Domain':<22} | {'N':>5} | "
          f"{'TR@1':>6} {'TR@5':>6} {'TR@10':>6} {'TnDCG@10':>9} | "
          f"{'WR@1':>6} {'WR@5':>6} {'WR@10':>6} {'WnDCG@10':>9}")
    print("-" * 130)

    domain_summary_rows = []
    for domain in all_domains:
        dm = all_domain_metrics[domain]

        t_m   = dm.get('traditional',   _init_metric())
        t_cnt = t_m['count'] if t_m['count'] > 0 else 1
        w_m   = dm.get('trad_weighted', _init_metric())
        w_cnt = w_m['count'] if w_m['count'] > 0 else 1

        print(
            f"{domain:<22} | {t_cnt:>5} | "
            f"{t_m['r1']/t_cnt*100:6.1f}% {t_m['r5']/t_cnt*100:6.1f}% "
            f"{t_m['r10']/t_cnt*100:6.1f}% {t_m['n10']/t_cnt:9.4f} | "
            f"{w_m['r1']/w_cnt*100:6.1f}% {w_m['r5']/w_cnt*100:6.1f}% "
            f"{w_m['r10']/w_cnt*100:6.1f}% {w_m['n10']/w_cnt:9.4f}"
        )

        row = {
            'domain':         domain,
            'n_queries':      t_cnt,
            'trad_r@1':       round(t_m['r1']  / t_cnt * 100, 4),
            'trad_r@5':       round(t_m['r5']  / t_cnt * 100, 4),
            'trad_r@10':      round(t_m['r10'] / t_cnt * 100, 4),
            'trad_ndcg@10':   round(t_m['n10'] / t_cnt,       6),
            'tradW_r@1':      round(w_m['r1']  / w_cnt * 100, 4),
            'tradW_r@5':      round(w_m['r5']  / w_cnt * 100, 4),
            'tradW_r@10':     round(w_m['r10'] / w_cnt * 100, 4),
            'tradW_ndcg@10':  round(w_m['n10'] / w_cnt,       6),
        }
        for key in ABLATION_KEYS:
            if key in ('traditional', 'trad_weighted'):
                continue
            m_   = dm.get(key, _init_metric())
            cnt_ = m_['count'] if m_['count'] > 0 else 1
            row[f'{key}_r@1']     = round(m_['r1']  / cnt_ * 100, 4)
            row[f'{key}_r@5']     = round(m_['r5']  / cnt_ * 100, 4)
            row[f'{key}_r@10']    = round(m_['r10'] / cnt_ * 100, 4)
            row[f'{key}_ndcg@10'] = round(m_['n10'] / cnt_,       6)
        domain_summary_rows.append(row)

    print("=" * 130)

    # Per-domain × n_last_layers table (norm=pre, r=0.5, weighted)
    print("\n--- Per-domain: n_last_layers comparison (nDCG@10, weighted scoring, norm=pre, r=0.5) ---")
    hdr_cols = [f"L{n}" for n in N_LAST_LAYERS_LIST]
    sep_len  = 22 + 3 + 8 + 3 + len(N_LAST_LAYERS_LIST) * 11
    print(f"{'Domain':<22} | {'Ratio':>6} | " + " | ".join(f"{c:>8}" for c in hdr_cols))
    print("-" * sep_len)

    for domain in all_domains:
        dm = all_domain_metrics[domain]
        first_row = True
        for r in TOPK_RATIOS:
            vals = []
            for n_layers in N_LAST_LAYERS_LIST:
                key  = f"L{n_layers}_normpre_r{int(r*100)}_weighted"
                m_   = dm.get(key, _init_metric())
                cnt_ = m_['count'] if m_['count'] > 0 else 1
                vals.append(m_['n10'] / cnt_)
            dom_label = domain if first_row else ""
            first_row = False
            print(f"{dom_label:<22} | {r:>6.1f} | " + " | ".join(f"{v:8.4f}" for v in vals))
        print("-" * sep_len)

    # Save CSVs
    domain_summary_df   = pd.DataFrame(domain_summary_rows)
    domain_summary_path = os.path.join(WORKING_DIR, "MASTER_domain_summary.csv")
    domain_summary_df.to_csv(domain_summary_path, index=False)
    print(f"\n✅ Domain summary → {domain_summary_path}")

    summary_rows = []
    for method in ABLATION_KEYS:
        if method not in all_metrics:
            continue
        m   = all_metrics[method]
        cnt = m['count'] if m['count'] > 0 else 1
        summary_rows.append({
            'method':  method,
            'r@1':     round(m['r1']  / cnt * 100, 4),
            'r@5':     round(m['r5']  / cnt * 100, 4),
            'r@10':    round(m['r10'] / cnt * 100, 4),
            'ndcg@1':  round(m['n1']  / cnt,       6),
            'ndcg@5':  round(m['n5']  / cnt,       6),
            'ndcg@10': round(m['n10'] / cnt,       6),
        })
    summary_df   = pd.DataFrame(summary_rows)
    summary_path = os.path.join(WORKING_DIR, "MASTER_ablation_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"✅ Ablation summary → {summary_path}")

    if all_batch_stats:
        pd.DataFrame(all_batch_stats).to_csv(
            os.path.join(WORKING_DIR, "MASTER_batch_stats.csv"), index=False
        )
        print("Batch stats:")
        for row in all_batch_stats:
            print(f"  {row}")

except Exception as e:
    print(f"❌ Final aggregation error: {e}")
    import traceback; traceback.print_exc()


# ==============================================================================
# BƯỚC 5: VISUALIZE ABLATION
# ==============================================================================
print("\n>>> BƯỚC 5: Visualize Ablation Results")

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    import numpy as np

    METRICS_PLOT = [
        ('ndcg@10',   'n10', 'nDCG@10',   False),
        ('recall@10', 'r10', 'Recall@10',  True),
        ('ndcg@5',    'n5',  'nDCG@5',    False),
        ('recall@1',  'r1',  'Recall@1',   True),
    ]

    COLORS_SCORING = {'trad': '#1D9E75', 'weighted': '#E07B39'}
    LABELS_SCORING = {'trad': 'Traditional sum', 'weighted': 'Importance-weighted'}
    COLORS_N       = {4: '#1D9E75', 8: '#E07B39', 16: '#3B72C4', 32: '#9B59B6'}
    COLORS_MODE    = {'pre': '#C23B22', 'post': '#4682B4'}

    def _mval(m, mkey, pct):
        cnt = m['count'] if m['count'] > 0 else 1
        v   = m[mkey] / cnt
        return v * 100 if pct else v

    # ---- Figure 1: per (n_layers, norm_mode) × metric ----
    n_variants = len(N_LAST_LAYERS_LIST) * len(NORMALIZE_MODES)
    n_metric   = len(METRICS_PLOT)

    fig, axes = plt.subplots(n_variants, n_metric,
                             figsize=(5 * n_metric, 4 * n_variants),
                             squeeze=False)
    row = 0
    for n_layers in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for col, (_, mkey, title, pct_flag) in enumerate(METRICS_PLOT):
                ax = axes[row][col]
                ax.set_title(f"L={n_layers}, norm={mode}\n{title}", fontsize=9)
                ax.set_xlabel('Top-k ratio', fontsize=8)
                ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=8)
                ax.grid(axis='y', alpha=0.25, linewidth=0.5)
                ax.spines[['top', 'right']].set_visible(False)

                if 'traditional' in all_metrics:
                    bv = _mval(all_metrics['traditional'], mkey, pct_flag)
                    ax.axhline(bv, color='#888780', linewidth=1.4,
                               linestyle='--', label='Baseline', zorder=2)

                for scoring in ['trad', 'weighted']:
                    vals = []
                    for r in TOPK_RATIOS:
                        tag = f"L{n_layers}_norm{mode}_r{int(r*100)}_{scoring}"
                        v   = _mval(all_metrics[tag], mkey, pct_flag) if tag in all_metrics else None
                        vals.append(v)
                    valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
                    if valid:
                        xs, ys = zip(*valid)
                        ax.plot(xs, ys, color=COLORS_SCORING[scoring],
                                linewidth=2.0, marker='o', markersize=4,
                                label=LABELS_SCORING[scoring], zorder=3)

                ax.set_xticks(TOPK_RATIOS)
                ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=7)
                fmt = (lambda v, _: f"{v:.1f}%") if pct_flag else (lambda v, _: f"{v:.3f}")
                ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
                if col == 0:
                    ax.legend(fontsize=7, framealpha=0.4, loc='lower right')
            row += 1

    plt.suptitle("Ablation: Cluster-Pool × n_last_layers × scoring mode", fontsize=11, y=1.005)
    plt.tight_layout()
    plot1 = os.path.join(WORKING_DIR, "ablation_per_variant.png")
    plt.savefig(plot1, dpi=140, bbox_inches='tight')
    plt.close()
    print(f"✅ Plot 1 → {plot1}")

    # ---- Figure 2: n_last_layers comparison ----
    fig2, axes2 = plt.subplots(1, len(NORMALIZE_MODES),
                               figsize=(7 * len(NORMALIZE_MODES), 5))
    if len(NORMALIZE_MODES) == 1:
        axes2 = [axes2]

    for ci, mode in enumerate(NORMALIZE_MODES):
        ax = axes2[ci]
        ax.set_title(f"norm={mode} — nDCG@10 (weighted scoring)", fontsize=10)
        ax.set_xlabel('Top-k ratio', fontsize=9)
        ax.set_ylabel('nDCG@10', fontsize=9)
        ax.grid(axis='y', alpha=0.25, linewidth=0.5)
        ax.spines[['top', 'right']].set_visible(False)

        if 'traditional' in all_metrics:
            bv = _mval(all_metrics['traditional'], 'n10', False)
            ax.axhline(bv, color='#888780', linewidth=1.4, linestyle='--', label='Baseline', zorder=2)

        for n_layers in N_LAST_LAYERS_LIST:
            vals = []
            for r in TOPK_RATIOS:
                tag = f"L{n_layers}_norm{mode}_r{int(r*100)}_weighted"
                v   = _mval(all_metrics[tag], 'n10', False) if tag in all_metrics else None
                vals.append(v)
            valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
            if valid:
                xs, ys = zip(*valid)
                ax.plot(xs, ys, color=COLORS_N.get(n_layers, 'black'),
                        linewidth=2.0, marker='o', markersize=4,
                        label=f"L={n_layers}", zorder=3)

        ax.set_xticks(TOPK_RATIOS)
        ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.3f}"))
        ax.legend(fontsize=8, framealpha=0.4)

    plt.suptitle("n_last_layers comparison (4/8/16/32) — Cluster-Pool Weighted Scoring", fontsize=11)
    plt.tight_layout()
    plot2 = os.path.join(WORKING_DIR, "ablation_nlayers_comparison.png")
    plt.savefig(plot2, dpi=140, bbox_inches='tight')
    plt.close()
    print(f"✅ Plot 2 → {plot2}")

    # ---- Figure 3: pre vs post normalize ----
    fig3, axes3 = plt.subplots(1, len(N_LAST_LAYERS_LIST),
                               figsize=(6 * len(N_LAST_LAYERS_LIST), 5))
    if len(N_LAST_LAYERS_LIST) == 1:
        axes3 = [axes3]

    for ci, n_layers in enumerate(N_LAST_LAYERS_LIST):
        ax = axes3[ci]
        ax.set_title(f"L={n_layers} — nDCG@10 (weighted scoring)", fontsize=10)
        ax.set_xlabel('Top-k ratio', fontsize=9)
        ax.set_ylabel('nDCG@10', fontsize=9)
        ax.grid(axis='y', alpha=0.25, linewidth=0.5)
        ax.spines[['top', 'right']].set_visible(False)

        if 'traditional' in all_metrics:
            bv = _mval(all_metrics['traditional'], 'n10', False)
            ax.axhline(bv, color='#888780', linewidth=1.4, linestyle='--', label='Baseline', zorder=2)

        for mode in NORMALIZE_MODES:
            vals = []
            for r in TOPK_RATIOS:
                tag = f"L{n_layers}_norm{mode}_r{int(r*100)}_weighted"
                v   = _mval(all_metrics[tag], 'n10', False) if tag in all_metrics else None
                vals.append(v)
            valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
            if valid:
                xs, ys = zip(*valid)
                ax.plot(xs, ys, color=COLORS_MODE[mode],
                        linewidth=2.0, marker='o', markersize=4,
                        label=f"norm={mode}", zorder=3)

        ax.set_xticks(TOPK_RATIOS)
        ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.3f}"))
        ax.legend(fontsize=8, framealpha=0.4)

    plt.suptitle("Pre-normalize vs Post-normalize — Cluster-Pool Weighted Scoring", fontsize=11)
    plt.tight_layout()
    plot3 = os.path.join(WORKING_DIR, "ablation_normalize_comparison.png")
    plt.savefig(plot3, dpi=140, bbox_inches='tight')
    plt.close()
    print(f"✅ Plot 3 → {plot3}")

    # ---- Figure 4: Per-domain heatmap (domain × n_last_layers) ----
    if all_domain_metrics:
        all_domains_sorted = sorted(all_domain_metrics.keys())
        n_dom = len(all_domains_sorted)
        n_lay = len(N_LAST_LAYERS_LIST)

        heatmap_data = np.zeros((n_dom, n_lay))
        for i, domain in enumerate(all_domains_sorted):
            dm = all_domain_metrics[domain]
            for j, n_layers in enumerate(N_LAST_LAYERS_LIST):
                key  = f"L{n_layers}_normpre_r50_weighted"
                m_   = dm.get(key, _init_metric())
                cnt_ = m_['count'] if m_['count'] > 0 else 1
                heatmap_data[i, j] = m_['n10'] / cnt_

        fig4, ax4 = plt.subplots(figsize=(max(6, n_lay * 1.8), max(4, n_dom * 0.7)))
        im   = ax4.imshow(heatmap_data, aspect='auto', cmap='YlGn')
        ax4.set_xticks(range(n_lay))
        ax4.set_xticklabels([f"L={n}" for n in N_LAST_LAYERS_LIST], fontsize=9)
        ax4.set_yticks(range(n_dom))
        ax4.set_yticklabels(all_domains_sorted, fontsize=8)
        ax4.set_xlabel('n_last_layers', fontsize=9)
        ax4.set_title('nDCG@10 by domain × n_last_layers\n(norm=pre, ratio=0.5, weighted scoring)', fontsize=10)
        plt.colorbar(im, ax=ax4, label='nDCG@10')
        vmax = heatmap_data.max()
        for i in range(n_dom):
            for j in range(n_lay):
                ax4.text(j, i, f"{heatmap_data[i,j]:.3f}",
                         ha='center', va='center', fontsize=7,
                         color='white' if heatmap_data[i, j] > vmax * 0.7 else 'black')
        plt.tight_layout()
        plot4 = os.path.join(WORKING_DIR, "ablation_domain_heatmap.png")
        plt.savefig(plot4, dpi=140, bbox_inches='tight')
        plt.close()
        print(f"✅ Plot 4 (domain heatmap) → {plot4}")

    # ---- Figure 5: Per-domain bar chart — traditional vs best ablation ----
    if all_domain_metrics:
        all_domains_sorted = sorted(all_domain_metrics.keys())
        n_dom    = len(all_domains_sorted)
        BEST_TAG = "L16_normpre_r50_weighted"

        trad_vals = []
        best_vals = []
        for domain in all_domains_sorted:
            dm    = all_domain_metrics[domain]
            t_m   = dm.get('traditional', _init_metric())
            t_cnt = t_m['count'] if t_m['count'] > 0 else 1
            trad_vals.append(t_m['n10'] / t_cnt)

            b_m   = dm.get(BEST_TAG, _init_metric())
            b_cnt = b_m['count'] if b_m['count'] > 0 else 1
            best_vals.append(b_m['n10'] / b_cnt)

        x     = np.arange(n_dom)
        width = 0.35
        fig5, ax5 = plt.subplots(figsize=(max(8, n_dom * 1.2), 5))
        ax5.bar(x - width/2, trad_vals, width, label='Traditional',  color='#888780', alpha=0.85)
        ax5.bar(x + width/2, best_vals, width, label=f'{BEST_TAG}',  color='#3B72C4', alpha=0.85)
        ax5.set_xlabel('Domain', fontsize=9)
        ax5.set_ylabel('nDCG@10', fontsize=9)
        ax5.set_title('Per-domain nDCG@10: Traditional vs Best Ablation Config', fontsize=10)
        ax5.set_xticks(x)
        ax5.set_xticklabels(all_domains_sorted, rotation=30, ha='right', fontsize=8)
        ax5.legend(fontsize=8, framealpha=0.4)
        ax5.grid(axis='y', alpha=0.25, linewidth=0.5)
        ax5.spines[['top', 'right']].set_visible(False)
        plt.tight_layout()
        plot5 = os.path.join(WORKING_DIR, "ablation_domain_bar.png")
        plt.savefig(plot5, dpi=140, bbox_inches='tight')
        plt.close()
        print(f"✅ Plot 5 (domain bar) → {plot5}")

except Exception as e:
    print(f"⚠️  Plot error (non-fatal): {e}")
    import traceback; traceback.print_exc()

print("\n>>> BƯỚC 4+5 COMPLETE — ABLATION STUDY DONE")